In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier,GradientBoostingClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.compose import make_column_selector, make_column_transformer

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, log_loss

In [19]:
df = pd.read_csv("Vehicle (2).csv")
df.head()

,Comp,Circ,D.Circ,Rad.Ra,Pr.Axis.Ra,Max.L.Ra,Scat.Ra,Elong,Pr.Axis.Rect,Max.L.Rect,Sc.Var.Maxis,Sc.Var.maxis,Ra.Gyr,Skew.Maxis,Skew.maxis,Kurt.maxis,Kurt.Maxis,Holl.Ra,Class
0,95,48,83,178,72,10,162,42,20,159,176,379,184,70,6,16,187,197,van
1,91,41,84,141,57,9,149,45,19,143,170,330,158,72,9,14,189,199,van
2,104,50,106,209,66,10,207,32,23,158,223,635,220,73,14,9,188,196,saab
3,93,41,82,159,63,9,144,46,19,143,160,309,127,63,6,10,199,207,van
4,85,44,70,205,103,52,149,45,19,144,241,325,188,127,9,11,180,183,bus


In [20]:
X,y = df.drop("Class",axis=1), df["Class"]
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=26,test_size=0.3,stratify=y)

In [21]:
ohc = OneHotEncoder(sparse_output=False, drop="first").set_output(transform="pandas")
ct = make_column_transformer((ohc,make_column_selector(dtype_include=object)),
                             ("passthrough", make_column_selector(dtype_include=['int64','float64'])),
                             verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')

In [22]:
lr = np.linspace(0.01,0.9,10)
max_depth = [None, 2, 3, 4]
n_estimators = [25, 50, 100]
scores = []
for l in tqdm(lr):
    for m in max_depth:
        for n in n_estimators:
            gdc = GradientBoostingClassifier(random_state=26,learning_rate=l, n_estimators= n, max_depth= m)
            gdc.fit(X_train,y_train)
            y_pred = gdc.predict_proba(X_test)
            scores.append([l,m,n,log_loss(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=["Learning Rate","Max Depth", "N", "Loss"])


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [01:28<00:00,  8.88s/it]


In [23]:
df_scores.sort_values("Loss")


,Learning Rate,Max Depth,N,Loss
63,0.504444,2.0,25,0.478272
52,0.405556,2.0,50,0.480414
17,0.108889,2.0,100,0.489373
51,0.405556,2.0,25,0.490844
40,0.306667,2.0,50,0.495539
...,...,...,...,...
109,0.900000,NaN,50,2.701495
108,0.900000,NaN,25,2.701495
86,0.702222,NaN,100,2.891586
85,0.702222,NaN,50,2.891586
